# Adaptive RAG with AWS

## 📖 Overview

This notebook demonstrates **Adaptive RAG** using AWS services:
- **Amazon Bedrock Claude** for query classification and routing
- **AWS OpenSearch Serverless** for vector storage
- **Amazon Bedrock Titan** for embeddings
- **Dynamic strategy selection** based on query characteristics

### What is Adaptive RAG?

Adaptive RAG **dynamically selects the best retrieval strategy** based on query type, rather than using one-size-fits-all approach.

**Problem with Static RAG:**
- Simple factual queries don't need expensive fusion retrieval
- Complex queries suffer with basic vector search
- All queries use the same strategy regardless of need
- Wastes resources on over-engineering simple queries

**Adaptive RAG Solution:**
```
Query: "What is AWS?" → Route to: Simple RAG (fast, cheap)
Query: "Compare AWS services for ML workloads" → Route to: Fusion + Reranking (thorough)
Query: "How is Bedrock related to SageMaker?" → Route to: Graph RAG (relationships)
```

### How It Works

1. **Classify**: Analyze query characteristics (complexity, type, intent)
2. **Route**: Select optimal RAG strategy
3. **Execute**: Run chosen strategy
4. **Learn**: Track which strategies work best (optional)

### When to Use

✅ **Good for:**
- Production systems with diverse queries
- Cost optimization (avoid over-engineering)
- Latency optimization (fast path for simple queries)
- Quality optimization (complex strategies when needed)
- Systems handling mixed query complexity

❌ **Not ideal for:**
- Uniform query types only
- Very simple applications
- When routing overhead too high
- Prototyping/development phase
- When one strategy already optimal

### Architecture

```mermaid
graph TB
    A[User Query] --> B[Query Classifier<br/>Claude]
    B --> C{Query Type?}
    
    C -->|Simple Factual| D1[Simple RAG<br/>Fast & Cheap]
    C -->|Complex/Ambiguous| D2[Fusion RAG<br/>Better Recall]
    C -->|Relationship| D3[Graph RAG<br/>Multi-hop]
    C -->|Precision Critical| D4[Reranking RAG<br/>High Quality]
    C -->|Vocabulary Mismatch| D5[HyDE RAG<br/>Bridge Gap]
    
    D1 --> E[Generate Answer]
    D2 --> E
    D3 --> E
    D4 --> E
    D5 --> E
    
    E --> F[Return Answer + Metadata]
    F --> G[Track Performance<br/>Learn & Optimize]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D1 fill:#c8e6c9
    style D2 fill:#fff9c4
    style D3 fill:#ffe0b2
    style D4 fill:#ffccbc
    style D5 fill:#e1bee7
    style E fill:#b3e5fc
    style F fill:#c8e6c9
    style G fill:#f0f4c3
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
from typing import List, Dict, Tuple, Optional
from enum import Enum
import time
from collections import defaultdict

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

## 2️⃣ Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
INDEX_NAME = 'adaptive_rag_docs'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
CLASSIFIER_MODEL = 'us.anthropic.claude-haiku-3-20241022-v1:0'  # Fast for routing
LLM_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'  # Quality for answers

# Strategy Configuration
ENABLE_LEARNING = True  # Track performance for optimization

print(f"Configuration:")
print(f"  Classifier: {CLASSIFIER_MODEL.split('.')[-1][:25]}")
print(f"  Answer LLM: {LLM_MODEL.split('.')[-1][:25]}")
print(f"  Learning enabled: {ENABLE_LEARNING}")

## 3️⃣ Define RAG Strategies

In [ ]:
class RAGStrategy(Enum):
    """Available RAG strategies"""
    SIMPLE = "simple"
    FUSION = "fusion"
    HYDE = "hyde"
    RERANKING = "reranking"
    COMPRESSION = "compression"

class QueryType(Enum):
    """Query classification types"""
    SIMPLE_FACTUAL = "simple_factual"
    COMPLEX_ANALYTICAL = "complex_analytical"
    COMPARISON = "comparison"
    RELATIONSHIP = "relationship"
    AMBIGUOUS = "ambiguous"
    DEFINITION = "definition"

# Strategy routing rules
STRATEGY_ROUTING = {
    QueryType.SIMPLE_FACTUAL: RAGStrategy.SIMPLE,
    QueryType.DEFINITION: RAGStrategy.SIMPLE,
    QueryType.COMPLEX_ANALYTICAL: RAGStrategy.FUSION,
    QueryType.COMPARISON: RAGStrategy.RERANKING,
    QueryType.RELATIONSHIP: RAGStrategy.SIMPLE,  # Would be GRAPH in full implementation
    QueryType.AMBIGUOUS: RAGStrategy.HYDE,
}

# Strategy characteristics
STRATEGY_SPECS = {
    RAGStrategy.SIMPLE: {
        'cost': 0.05,
        'latency': 1.5,
        'quality': 7,
        'description': 'Fast vector search + generation'
    },
    RAGStrategy.FUSION: {
        'cost': 0.065,
        'latency': 3.5,
        'quality': 8.5,
        'description': 'Multi-query with RRF fusion'
    },
    RAGStrategy.HYDE: {
        'cost': 0.051,
        'latency': 2.0,
        'quality': 8,
        'description': 'Hypothetical document embeddings'
    },
    RAGStrategy.RERANKING: {
        'cost': 0.056,
        'latency': 2.5,
        'quality': 9,
        'description': 'Two-stage with LLM reranking'
    },
    RAGStrategy.COMPRESSION: {
        'cost': 0.048,
        'latency': 2.8,
        'quality': 8.5,
        'description': 'Context compression'
    },
}

print("Available RAG Strategies:\n")
for strategy, specs in STRATEGY_SPECS.items():
    print(f"  {strategy.value:12} - {specs['description']}")
    print(f"               Cost: ${specs['cost']:.3f}, Latency: {specs['latency']:.1f}s, Quality: {specs['quality']}/10")

## 4️⃣ Initialize Services

In [ ]:
opensearch = OpenSearchManager(region=AWS_REGION, index_name=INDEX_NAME)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
classifier = BedrockLLM(AWS_REGION, CLASSIFIER_MODEL, temperature=0.1)
llm = BedrockLLM(AWS_REGION, LLM_MODEL, temperature=0.7)

# Performance tracking
performance_tracker = defaultdict(lambda: {'count': 0, 'total_time': 0, 'successes': 0})

print("✓ Services initialized")

## 5️⃣ Load and Index Sample Documents

In [ ]:
sample_documents = [
    "AWS Bedrock is a fully managed service providing access to foundation models from leading AI companies through a single API.",
    "Claude Opus is Anthropic's most capable model, excelling at complex analysis and multi-step reasoning tasks.",
    "Retrieval-Augmented Generation (RAG) combines information retrieval with text generation to provide grounded, factual responses.",
    "Amazon Titan Embeddings V2 generates 1024-dimensional vectors optimized for semantic search and clustering.",
    "OpenSearch Serverless automatically scales compute and storage, eliminating cluster management overhead.",
    "Vector search enables finding semantically similar content by comparing embeddings in high-dimensional space.",
    "HyDE improves retrieval by generating hypothetical answers and using them for similarity search.",
    "Reranking uses LLMs to score relevance of retrieved documents, improving precision over pure vector search.",
    "Fusion retrieval generates multiple query variations and merges results using reciprocal rank fusion.",
    "Contextual compression extracts only relevant snippets from documents, reducing tokens while preserving information.",
    "Semantic chunking creates boundaries based on meaning rather than fixed character counts.",
    "Adaptive RAG routes queries to optimal strategies based on query type and complexity.",
    "Graph RAG builds knowledge graphs enabling multi-hop reasoning and relationship queries.",
    "Production RAG systems require monitoring metrics like precision, recall, latency, and cost.",
    "Caching embeddings and results significantly reduces costs for frequently accessed queries."
]

# Create index and index documents
opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

print("Indexing documents...")
documents = []
for i, text in enumerate(sample_documents):
    embedding = embedder.embed_text(text)
    documents.append({
        'text': text,
        'embedding': embedding,
        'metadata': {'doc_id': i}
    })

opensearch.index_documents(documents)
print(f"✓ Indexed {len(documents)} documents")

## 6️⃣ Query Classifier

Classify queries into types for routing.

### Classification Criteria

**Simple Factual**: "What is X?", "Define Y"
- Single concept
- Clear definition sought
- Direct answer expected

**Complex Analytical**: "Explain how...", "What are the implications..."
- Multi-faceted
- Requires synthesis
- Deep analysis needed

**Comparison**: "Compare X and Y", "What's the difference..."
- Multiple entities
- Contrast needed
- Precision important

**Relationship**: "How is X related to Y?"
- Connections sought
- Graph traversal helpful

**Ambiguous**: Vague or unclear queries
- Multiple interpretations
- Needs disambiguation
- HyDE helps

In [ ]:
def classify_query(query: str) -> QueryType:
    """
    Classify query type using LLM
    
    Returns:
        QueryType enum
    """
    classification_prompt = f"""
Classify the following query into ONE of these categories:

Categories:
1. SIMPLE_FACTUAL - Simple "what is" or definition questions
2. DEFINITION - Asking for a definition or explanation of a concept
3. COMPLEX_ANALYTICAL - Multi-faceted questions requiring deep analysis
4. COMPARISON - Comparing two or more things
5. RELATIONSHIP - Asking about relationships or connections
6. AMBIGUOUS - Vague or unclear questions

Query: {query}

Respond with ONLY the category name (e.g., "SIMPLE_FACTUAL"), nothing else.
"""
    
    try:
        response = classifier.generate(classification_prompt, temperature=0.1)
        response = response.strip().upper()
        
        # Map response to QueryType
        for query_type in QueryType:
            if query_type.name in response:
                return query_type
        
        # Default fallback
        return QueryType.SIMPLE_FACTUAL
    
    except Exception as e:
        print(f"Classification error: {e}")
        return QueryType.SIMPLE_FACTUAL

# Test classifier
test_queries_for_classification = [
    "What is AWS Bedrock?",
    "Compare RAG with traditional LLM approaches",
    "How is Claude related to AWS?",
    "Explain the implications of using vector search",
    "Tell me about search",
]

print("Query Classification Test:\n" + "="*70 + "\n")
for query in test_queries_for_classification:
    query_type = classify_query(query)
    strategy = STRATEGY_ROUTING[query_type]
    print(f"Query: {query}")
    print(f"  → Type: {query_type.value}")
    print(f"  → Strategy: {strategy.value}")
    print()

## 7️⃣ Implement RAG Strategies

Simplified implementations of each strategy.

In [ ]:
def simple_rag(query: str, top_k: int = 5) -> Tuple[str, List[Dict]]:
    """Simple vector search + generation"""
    query_emb = embedder.embed_text(query)
    results = opensearch.vector_search(query_emb, top_k=top_k)
    answer = llm.generate_with_context(query, [r['text'] for r in results])
    return answer, results

def fusion_rag(query: str, top_k: int = 5) -> Tuple[str, List[Dict]]:
    """Multi-query fusion (simplified)"""
    # Generate query variants
    variant_prompt = f"Generate 2 alternative phrasings of: {query}\nList them numbered 1-2, no explanations."
    variants_text = llm.generate(variant_prompt, temperature=0.9)
    variants = [query] + [line.split('. ', 1)[-1].strip() for line in variants_text.split('\n') if line.strip() and line[0].isdigit()][:2]
    
    # Search with each variant
    all_results = []
    for variant in variants:
        emb = embedder.embed_text(variant)
        results = opensearch.vector_search(emb, top_k=top_k)
        all_results.extend(results)
    
    # Simple deduplication by doc_id
    seen = set()
    unique_results = []
    for r in all_results:
        doc_id = r['metadata']['doc_id']
        if doc_id not in seen:
            seen.add(doc_id)
            unique_results.append(r)
    
    answer = llm.generate_with_context(query, [r['text'] for r in unique_results[:top_k]])
    return answer, unique_results[:top_k]

def hyde_rag(query: str, top_k: int = 5) -> Tuple[str, List[Dict]]:
    """HyDE approach"""
    # Generate hypothetical answer
    hyp_prompt = f"Write a detailed answer to: {query}\nAnswer:"
    hypothesis = llm.generate(hyp_prompt, temperature=0.7)
    
    # Search with hypothesis
    hyp_emb = embedder.embed_text(hypothesis)
    results = opensearch.vector_search(hyp_emb, top_k=top_k)
    
    # Generate real answer
    answer = llm.generate_with_context(query, [r['text'] for r in results])
    return answer, results

def reranking_rag(query: str, top_k: int = 5) -> Tuple[str, List[Dict]]:
    """Retrieval with reranking (simplified)"""
    # Retrieve more candidates
    query_emb = embedder.embed_text(query)
    candidates = opensearch.vector_search(query_emb, top_k=top_k * 2)
    
    # Simple reranking by asking LLM
    rerank_prompt = f"""Rate relevance of these documents to query "{query}" on scale 0-10.
Return ONLY comma-separated scores.

Documents:
""" + "\n".join([f"{i+1}. {doc['text'][:100]}..." for i, doc in enumerate(candidates)])
    
    scores_text = classifier.generate(rerank_prompt, temperature=0.1)
    try:
        scores = [float(s.strip()) for s in scores_text.split(',')[:len(candidates)]]
        # Sort by score
        scored = list(zip(candidates, scores))
        scored.sort(key=lambda x: x[1], reverse=True)
        results = [doc for doc, score in scored[:top_k]]
    except:
        results = candidates[:top_k]
    
    answer = llm.generate_with_context(query, [r['text'] for r in results])
    return answer, results

def compression_rag(query: str, top_k: int = 5) -> Tuple[str, List[Dict]]:
    """Retrieval with compression (simplified)"""
    query_emb = embedder.embed_text(query)
    candidates = opensearch.vector_search(query_emb, top_k=top_k * 2)
    
    # Compress each document
    compressed = []
    for doc in candidates:
        comp_prompt = f"Extract ONLY parts relevant to '{query}':\n{doc['text']}\n\nRelevant parts:"
        compressed_text = classifier.generate(comp_prompt, temperature=0.1)
        if len(compressed_text.split()) > 10:
            compressed.append({'text': compressed_text, **doc})
    
    results = compressed[:top_k]
    answer = llm.generate_with_context(query, [r['text'] for r in results])
    return answer, results

# Strategy executor mapping
STRATEGY_EXECUTORS = {
    RAGStrategy.SIMPLE: simple_rag,
    RAGStrategy.FUSION: fusion_rag,
    RAGStrategy.HYDE: hyde_rag,
    RAGStrategy.RERANKING: reranking_rag,
    RAGStrategy.COMPRESSION: compression_rag,
}

print("✓ RAG strategy implementations ready")

## 8️⃣ Adaptive RAG System

Complete adaptive routing system.

In [ ]:
def adaptive_rag(query: str, top_k: int = 5) -> Dict:
    """
    Adaptive RAG with intelligent routing
    
    Returns:
        Dict with answer, strategy used, metadata
    """
    start_time = time.time()
    
    # Step 1: Classify query
    print(f"Step 1: Classifying query...")
    classify_start = time.time()
    query_type = classify_query(query)
    classify_time = time.time() - classify_start
    print(f"✓ Classified as: {query_type.value} ({classify_time:.2f}s)\n")
    
    # Step 2: Route to strategy
    strategy = STRATEGY_ROUTING[query_type]
    print(f"Step 2: Routing to strategy: {strategy.value}")
    print(f"  {STRATEGY_SPECS[strategy]['description']}")
    print(f"  Expected: ${STRATEGY_SPECS[strategy]['cost']:.3f}, {STRATEGY_SPECS[strategy]['latency']:.1f}s\n")
    
    # Step 3: Execute strategy
    print(f"Step 3: Executing {strategy.value} strategy...")
    execute_start = time.time()
    executor = STRATEGY_EXECUTORS[strategy]
    answer, results = executor(query, top_k=top_k)
    execute_time = time.time() - execute_start
    print(f"✓ Strategy executed ({execute_time:.2f}s)\n")
    
    total_time = time.time() - start_time
    
    # Track performance
    if ENABLE_LEARNING:
        performance_tracker[strategy.value]['count'] += 1
        performance_tracker[strategy.value]['total_time'] += total_time
        performance_tracker[strategy.value]['successes'] += 1  # Simplified - would track actual quality
    
    print(f"Total time: {total_time:.2f}s")
    
    return {
        'answer': answer,
        'query_type': query_type.value,
        'strategy': strategy.value,
        'results': results,
        'metadata': {
            'classification_time': classify_time,
            'execution_time': execute_time,
            'total_time': total_time,
            'expected_cost': STRATEGY_SPECS[strategy]['cost'],
            'expected_quality': STRATEGY_SPECS[strategy]['quality']
        }
    }

# Test adaptive RAG
test_queries = [
    "What is AWS Bedrock?",
    "Compare vector search and keyword search approaches",
    "How does RAG improve LLM responses?",
    "Explain search optimization",
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*70}")
    print(f"Query {i}: {query}")
    print('='*70 + "\n")
    
    result = adaptive_rag(query, top_k=3)
    
    print(f"\n📊 Results:")
    print(f"  Query Type: {result['query_type']}")
    print(f"  Strategy Used: {result['strategy']}")
    print(f"  Retrieved: {len(result['results'])} documents")
    print(f"\n⏱️  Performance:")
    print(f"  Classification: {result['metadata']['classification_time']:.2f}s")
    print(f"  Execution: {result['metadata']['execution_time']:.2f}s")
    print(f"  Total: {result['metadata']['total_time']:.2f}s")
    print(f"\n💰 Cost: ${result['metadata']['expected_cost']:.3f}")
    print(f"📈 Quality Score: {result['metadata']['expected_quality']}/10")
    print(f"\n✅ Answer:\n{result['answer'][:200]}...")

## 9️⃣ Comparison: Adaptive vs Static Strategy

Compare adaptive routing against always using one strategy.

In [ ]:
# Test diverse queries
diverse_queries = [
    "What is RAG?",  # Simple
    "Compare Claude Opus and Haiku",  # Comparison
    "How can vector search be optimized for production?",  # Complex
    "Tell me about embeddings",  # Ambiguous
    "Define semantic search",  # Definition
]

print("Comparison: Adaptive vs Always-Simple Strategy\n")
print("="*70 + "\n")

adaptive_total_time = 0
adaptive_total_cost = 0
simple_total_time = 0
simple_total_cost = 0

for i, query in enumerate(diverse_queries, 1):
    print(f"Query {i}: {query}")
    
    # Adaptive
    result = adaptive_rag(query, top_k=3)
    adaptive_total_time += result['metadata']['total_time']
    adaptive_total_cost += result['metadata']['expected_cost']
    
    print(f"  Adaptive: {result['strategy']} - {result['metadata']['total_time']:.2f}s, ${result['metadata']['expected_cost']:.3f}")
    
    # Simple (baseline)
    simple_start = time.time()
    simple_answer, simple_results = simple_rag(query, top_k=3)
    simple_time = time.time() - simple_start
    simple_cost = STRATEGY_SPECS[RAGStrategy.SIMPLE]['cost']
    
    simple_total_time += simple_time
    simple_total_cost += simple_cost
    
    print(f"  Simple:   simple - {simple_time:.2f}s, ${simple_cost:.3f}")
    print()

print("="*70)
print(f"\n📊 TOTALS ({len(diverse_queries)} queries):\n")
print(f"  Adaptive RAG:")
print(f"    Total Time: {adaptive_total_time:.2f}s")
print(f"    Total Cost: ${adaptive_total_cost:.3f}")
print(f"    Avg Time:   {adaptive_total_time/len(diverse_queries):.2f}s per query")
print(f"    Avg Cost:   ${adaptive_total_cost/len(diverse_queries):.3f} per query")

print(f"\n  Static (Always Simple):")
print(f"    Total Time: {simple_total_time:.2f}s")
print(f"    Total Cost: ${simple_total_cost:.3f}")
print(f"    Avg Time:   {simple_total_time/len(diverse_queries):.2f}s per query")
print(f"    Avg Cost:   ${simple_total_cost/len(diverse_queries):.3f} per query")

print(f"\n💡 Analysis:")
print(f"  - Adaptive may be slightly slower due to routing overhead")
print(f"  - But provides better quality for complex queries")
print(f"  - Cost difference: ${adaptive_total_cost - simple_total_cost:.3f} ({(adaptive_total_cost/simple_total_cost - 1)*100:+.1f}%)")
print(f"  - Trade-off: Spend {(adaptive_total_time/simple_total_time - 1)*100:+.1f}% more time for better quality")

## 🔟 Performance Analytics

Analyze strategy usage and performance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("Strategy Usage Analytics\n")
print("="*70 + "\n")

if performance_tracker:
    strategies = list(performance_tracker.keys())
    counts = [performance_tracker[s]['count'] for s in strategies]
    avg_times = [performance_tracker[s]['total_time'] / performance_tracker[s]['count'] 
                 if performance_tracker[s]['count'] > 0 else 0 for s in strategies]
    
    print("Strategy Performance Summary:\n")
    for strategy in strategies:
        stats = performance_tracker[strategy]
        if stats['count'] > 0:
            print(f"  {strategy}:")
            print(f"    Times used: {stats['count']}")
            print(f"    Avg time: {stats['total_time']/stats['count']:.2f}s")
            print(f"    Success rate: {stats['successes']/stats['count']*100:.0f}%")
            print()
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Usage distribution
    ax1.pie(counts, labels=strategies, autopct='%1.1f%%', startangle=90)
    ax1.set_title('Strategy Usage Distribution', fontsize=14, fontweight='bold')
    
    # Average latency
    bars = ax2.bar(strategies, avg_times, color='skyblue', edgecolor='navy', linewidth=2)
    ax2.set_xlabel('Strategy', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Average Latency (seconds)', fontsize=12, fontweight='bold')
    ax2.set_title('Average Latency by Strategy', fontsize=14, fontweight='bold')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, time_val in zip(bars, avg_times):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{time_val:.2f}s',
                ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("No performance data collected yet")

print("\n💡 Insights:")
print("  - Track which strategies are used most often")
print("  - Identify slow strategies for optimization")
print("  - Adjust routing rules based on actual performance")
print("  - A/B test different routing strategies")

## 1️⃣1️⃣ Summary & Key Takeaways

### What We Built

✅ Query classification system
✅ Multiple RAG strategy implementations
✅ Intelligent routing logic
✅ Performance tracking and analytics
✅ Comparison with static approaches

### Performance Characteristics

**Benefits:**
- **Cost Optimization**: Use cheap strategies for simple queries
- **Latency Optimization**: Fast path for straightforward questions
- **Quality Optimization**: Complex strategies when needed
- **Resource Efficiency**: Right tool for each job

**Trade-offs:**
- **Routing Overhead**: ~0.2-0.5s for classification
- **Complexity**: More moving parts to maintain
- **Classification Errors**: Wrong routing = suboptimal results

### When to Use Adaptive RAG

**Use Adaptive when:**
- Query complexity varies widely
- Production system with cost concerns
- Need to balance speed, cost, and quality
- Multiple user personas
- Can track and learn from performance

**Skip Adaptive when:**
- Queries are uniform
- One strategy already optimal
- Routing overhead too high
- Prototyping phase
- Very simple use case

### Key Insights

1. **No One-Size-Fits-All**: Different queries need different strategies
2. **Routing is Fast**: Classification adds minimal overhead (0.2-0.5s)
3. **Track Performance**: Learning from actual usage improves routing
4. **Cost Savings**: Simple strategy for simple queries saves money
5. **Quality When Needed**: Complex strategies only for complex queries

### Routing Strategies

**1. Rule-Based (This Notebook)**
- Map query types to strategies
- Simple and interpretable
- Requires manual tuning

**2. LLM-Based**
- LLM chooses strategy directly
- More flexible
- Higher cost

**3. Learned Routing**
- Train classifier on historical data
- Best performance
- Requires training data

**4. Multi-Factor**
- Consider query + user + context
- Most sophisticated
- Complex to implement

### Best Practices

1. **Start Simple**: Begin with basic routing rules
2. **Track Metrics**: Monitor which strategies work best
3. **A/B Test**: Test routing changes on subset of traffic
4. **User Feedback**: Collect thumbs up/down to improve
5. **Fallback Strategy**: Default to safe strategy on errors

### Advanced Optimizations

**Caching:**
```python
# Cache classifications for similar queries
classification_cache = {}
cache_key = hash(query.lower())
if cache_key in classification_cache:
    return classification_cache[cache_key]
```

**Confidence Scoring:**
```python
# Route to more expensive strategy if low confidence
if classification_confidence < 0.7:
    strategy = RAGStrategy.FUSION  # More robust
```

**User Preferences:**
```python
# Allow users to prefer speed vs quality
if user_preference == "fast":
    strategy = RAGStrategy.SIMPLE
elif user_preference == "quality":
    strategy = RAGStrategy.RERANKING
```

**Dynamic Thresholds:**
```python
# Adjust routing based on system load
if system_load > 0.8:
    # Route more to simple strategies
    pass
```

### Limitations

1. **Classification Errors**: Wrong routing leads to suboptimal results
2. **Added Complexity**: More code to maintain and debug
3. **Routing Overhead**: Every query pays classification cost
4. **Strategy Maintenance**: Must keep all strategies updated
5. **Cold Start**: Need data to optimize routing

### Combining with Other Patterns

**Adaptive + Caching:**
- Cache results per strategy
- Faster repeated queries

**Adaptive + Monitoring:**
- Track strategy success rates
- Auto-adjust routing rules

**Adaptive + Fallback:**
- If strategy fails, try another
- Graceful degradation

### Metrics to Track

1. **Strategy Distribution**: Which strategies used most
2. **Success Rate**: By strategy
3. **Latency**: Per strategy
4. **Cost**: Actual vs expected
5. **Quality**: User feedback per strategy
6. **Classification Accuracy**: Are routes correct?

### Future Enhancements

- **Ensemble Routing**: Try multiple strategies, pick best
- **Reinforcement Learning**: Learn optimal routing policy
- **Context-Aware**: Consider conversation history
- **Multi-Stage**: Route different parts of pipeline
- **Confidence Calibration**: Better classification confidence

### Production Considerations

**Monitoring:**
- Track routing decisions
- Alert on unexpected distributions
- Monitor strategy success rates

**Testing:**
- Unit test each strategy
- Integration test routing logic
- A/B test routing changes

**Deployment:**
- Canary routing rule changes
- Feature flags for strategies
- Rollback capability

### Next Steps

- **Add more strategies**: Expand routing options
- **Improve classifier**: Fine-tune on domain data
- **Multi-factor routing**: User + query + context
- **Learning system**: Auto-optimize from feedback
- **Confidence scores**: Route based on certainty

## 🧹 Cleanup

In [ ]:
# Uncomment to delete index
# opensearch.delete_index(INDEX_NAME)
# print(f"✓ Deleted index: {INDEX_NAME}")

print("\nTo delete the index later:")
print(f"  opensearch.delete_index('{INDEX_NAME}')")

print("\n📊 Final Performance Summary:")
if performance_tracker:
    for strategy, stats in performance_tracker.items():
        if stats['count'] > 0:
            print(f"  {strategy}: {stats['count']} uses, {stats['total_time']/stats['count']:.2f}s avg")